In [55]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.visualization import simple_norm
from astropy.modeling.models import Sersic2D
from astropy.convolution import convolve_fft
from photutils.segmentation import detect_threshold, detect_sources
import time
from astropy.visualization import simple_norm
import statmorph
from statmorph.utils.image_diagnostics import make_figure
%matplotlib inline

# Python interpreter path: /nvme/scratch/software/anaconda3/envs/statmorph_env/bin/python

OUTPUT_DIR = "/nvme/scratch/work/alberttg/Summer_project/Data_products/Statmorph_fits"
CUTOUTS_DIR = "/nvme/scratch/work/alberttg/Summer_project/Data_products/Cutouts_3p0as"
PSF_DIR = "/nvme/scratch/work/alberttg/Summer_project/Data_products/PSFs"

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
    data = hdul[1].data
TABLE = Table(data)
GALAXY_ID = TABLE["SURVEY_ID"]
SURVEY = TABLE["SURVEY"]

FILTERS = ["F444W", "F356W", "F277W"]

In [56]:
def read_fits_path(path, ext):

    path = Path(path)

    if not path.exists():
        print(f"{path} not found.")
        return None

    try:
        with fits.open(path) as hdul:
            return hdul[ext].data
    except Exception as e:
        print(f"Could not open {path}: {e}")
        return None

In [57]:
def load_science_data(science_fits_path):
    """
    Load science image, segmentation-derived mask, and rms map from a
    multi-extension FITS file.

    ext[1] = science image
    ext[2] = segmentation map -> converted into a boolean mask
    ext[3] = rms / error map
    """
    with fits.open(science_fits_path) as hdul:
        image = hdul[1].data.astype(float)
        segmap = hdul[2].data
        rms = hdul[3].data.astype(float)

    if image.shape != segmap.shape or image.shape != rms.shape:
        raise ValueError(
            f"Shape mismatch: image {image.shape}, segmap {segmap.shape}, "
            f"rms {rms.shape}. All three extensions must match."
        )

    mask = build_mask_from_segmap(segmap)
    return image, mask, rms, segmap


def build_mask_from_segmap(segmap):
    """
    Convert a segmentation map into a boolean mask suitable for pysersic,
    where True = pixel should be masked/ignored (bad pixel or nearby
    contaminating source), and False = good pixel to use in the fit.

    Assumes the target galaxy is the segmentation ID present at the center
    of the cutout. All other nonzero IDs are masked out; background (0)
    stays unmasked.
    """
    ny, nx = segmap.shape
    cy, cx = ny // 2, nx // 2
    center_id = segmap[cy, cx]

    if center_id == 0:
        # Center pixel is background; search a small box around the center
        # for the nearest nonzero segmentation ID to use as the target.
        box = 5
        y0, y1 = max(0, cy - box), min(ny, cy + box + 1)
        x0, x1 = max(0, cx - box), min(nx, cx + box + 1)
        sub = segmap[y0:y1, x0:x1]
        nonzero = sub[sub != 0]
        if nonzero.size > 0:
            vals, counts = np.unique(nonzero, return_counts=True)
            center_id = vals[np.argmax(counts)]
        else:
            center_id = 0  # give up, nothing to mask as "target"

    # Mask everything that is a source (nonzero) and is NOT the target ID.
    mask = (segmap != 0) & (segmap != center_id)
    return mask


def load_and_crop_psf(psf_fits_path, science_image_shape, psf_ext=0):
    """
    Load the PSF from a FITS file and crop it (centered) so its dimensions
    are odd and no larger than the science image, which is the standard
    requirement for pysersic's convolution.
    """
    with fits.open(psf_fits_path) as hdul:
        psf = hdul[psf_ext].data.astype(float)

    psf = crop_to_odd(psf)

    # Also ensure PSF isn't larger than the science image itself.
    max_ny, max_nx = science_image_shape
    py, px = psf.shape
    target_y = min(py, max_ny if max_ny % 2 == 1 else max_ny - 1)
    target_x = min(px, max_nx if max_nx % 2 == 1 else max_nx - 1)

    if (target_y, target_x) != (py, px):
        psf = center_crop(psf, (target_y, target_x))
        psf = crop_to_odd(psf)

    # Normalize PSF to sum to 1
    psf = psf / np.nansum(psf)
    return psf


def crop_to_odd(arr):
    """Center-crop a 2D array so both dimensions are odd."""
    ny, nx = arr.shape
    new_ny = ny if ny % 2 == 1 else ny - 1
    new_nx = nx if nx % 2 == 1 else nx - 1
    if (new_ny, new_nx) != (ny, nx):
        arr = center_crop(arr, (new_ny, new_nx))
    return arr


def center_crop(arr, target_shape):
    """Center-crop a 2D array to the given target shape."""
    ny, nx = arr.shape
    ty, tx = target_shape
    y0 = (ny - ty) // 2
    x0 = (nx - tx) // 2
    return arr[y0:y0 + ty, x0:x0 + tx]

In [58]:
def _first_existing_attr(obj, names):
    """Return (name, value) for the first attribute in `names` that exists
    and is not None on obj, else (None, None)."""
    for name in names:
        if hasattr(obj, name):
            val = getattr(obj, name)
            if val is not None:
                return name, val
    return None, None


def _resolve_position_guess(props, image_shape):
    """
    Get an (x, y) center guess from a SourceProperties instance, trying
    several attribute names since this isn't consistent across pysersic
    versions (e.g. some expose `position_guess`, others split it into
    separate x/y attributes, others only via the underlying `cat` catalog
    object from photutils' data_properties/SourceCatalog).
    """
    name, val = _first_existing_attr(
        props, ["position_guess", "pos_guess", "xy_guess"]
    )
    if val is not None:
        return float(val[0]), float(val[1])

    xname, xval = _first_existing_attr(props, ["x_guess", "xc_guess", "x0_guess"])
    yname, yval = _first_existing_attr(props, ["y_guess", "yc_guess", "y0_guess"])
    if xval is not None and yval is not None:
        return float(xval), float(yval)

    # Fall back to the underlying photutils catalog object, if present.
    cat = getattr(props, "cat", None)
    if cat is not None:
        for xattr, yattr in [("xcentroid", "ycentroid"), ("x_centroid", "y_centroid")]:
            if hasattr(cat, xattr) and hasattr(cat, yattr):
                xv, yv = getattr(cat, xattr), getattr(cat, yattr)
                try:
                    return float(np.asarray(xv).ravel()[0]), float(np.asarray(yv).ravel()[0])
                except Exception:
                    return float(xv), float(yv)

    # Last resort: image center.
    ny, nx = image_shape
    print("    [warning] Could not find a position guess on SourceProperties "
          "or its .cat catalog; falling back to the image center. Run "
          "inspect_source_properties(props) to see what's actually "
          "available on your installed version.")
    return float(nx // 2), float(ny // 2)


def inspect_source_properties(props):
    """
    Diagnostic: print every public, non-callable attribute on a
    SourceProperties instance and its value, so you can see exactly what
    your installed pysersic version calls things (position/flux/r_eff/sky
    guesses etc.) instead of guessing attribute names blind.

        from run_pysersic_fit import SourceProperties, inspect_source_properties
        props = SourceProperties(image, mask=mask)
        inspect_source_properties(props)
    """
    print("Public attributes on this SourceProperties instance:")
    for name in sorted(dir(props)):
        if name.startswith("_"):
            continue
        try:
            val = getattr(props, name)
        except Exception as e:
            print(f"  {name}: <error accessing: {e}>")
            continue
        if callable(val):
            continue
        print(f"  {name} = {val!r}")


def _get_prop_guesses(props, image_shape):
    flux_name, flux_guess = _first_existing_attr(props, ["flux_guess"])
    _, flux_guess_err = _first_existing_attr(props, ["flux_guess_err"])
    _, r_eff_guess = _first_existing_attr(props, ["r_eff_guess"])
    _, r_eff_guess_err = _first_existing_attr(props, ["r_eff_guess_err"])
    _, sky_guess = _first_existing_attr(props, ["sky_guess"])
    _, sky_guess_err = _first_existing_attr(props, ["sky_guess_err"])

    missing = [n for n, v in [
        ("flux_guess", flux_guess), ("flux_guess_err", flux_guess_err),
        ("r_eff_guess", r_eff_guess), ("r_eff_guess_err", r_eff_guess_err),
        ("sky_guess", sky_guess), ("sky_guess_err", sky_guess_err),
    ] if v is None]
    if missing:
        raise AttributeError(
            f"SourceProperties is missing expected attribute(s): {missing}. "
            f"Run inspect_source_properties(props) on your installed "
            f"pysersic version to find the correct attribute names, then "
            f"update _get_prop_guesses() accordingly."
        )

    xg, yg = _resolve_position_guess(props, image_shape)

    return dict(
        flux_guess=float(flux_guess),
        flux_guess_err=float(flux_guess_err),
        position_guess=(xg, yg),
        r_eff_guess=float(r_eff_guess),
        r_eff_guess_err=float(r_eff_guess_err),
        sky_guess=float(sky_guess),
        sky_guess_err=float(sky_guess_err),
    )


def _find_map_svi(model, model_kwargs, rkey, num_steps=6000, learning_rate=3e-2):
    """
    Quick MAP-like point estimate for the custom model, via SVI with an
    AutoDelta guide (equivalent to MAP under a flat reference measure).
    Mirrors the role of fitter.find_MAP() for the other, natively
    supported profiles.

    Uses init_to_median() rather than numpyro's default init_to_uniform():
    the default samples a random starting point in *unconstrained* space,
    which for tightly-constrained priors (e.g. flux, whose sigma is often
    only ~1-2% of its mean) can start optimization miles from anything
    sensible -- a classic cause of "converges to garbage" for a model like
    this one with a flux<->point-source-fraction degeneracy. init_to_median
    starts at each prior's median instead, which is centered on the
    SourceProperties-derived guesses -- a much saner starting point.
    """
    guide = AutoDelta(model, init_loc_fn=init_to_median())
    svi = SVI(model, guide, Adam(learning_rate), loss=Trace_ELBO())
    svi_state = svi.init(rkey, **model_kwargs)

    def body(state, _):
        state, loss = svi.update(state, **model_kwargs)
        return state, loss

    svi_state, losses = jax.lax.scan(body, svi_state, None, length=num_steps)
    params = svi.get_params(svi_state)
    map_params = {k.replace("_auto_loc", ""): float(v) for k, v in params.items()}
    losses = np.asarray(losses)
    print(f"    SVI loss: start={losses[0]:.4e}  end={losses[-1]:.4e}  "
          f"min={losses.min():.4e}")
    if not np.isfinite(losses[-1]):
        print("    [warning] Final SVI loss is not finite -- MAP estimate "
              "is unreliable. Check the printed guesses/priors below for "
              "anything degenerate (e.g. zero or negative error bars).")
    return map_params, float(losses[-1])

In [59]:
def statmorph_model(image, segmap, psf, rms, mask):

    start = time.time()
    source_morphs = statmorph.source_morphology(
        image, segmap, psf=psf, weightmap=rms)
    print('Time: %g s.' % (time.time() - start))

    return source_morphs



In [60]:
def reduced_chi_squared(image, model, rms, mask, n_params=7):
    """
    Calulcates reduced chi squared between model (which is a convolution of psf and raw sersic model profile) and science image.
    n_params is 7: amplitude, sersic_rhalf, sersic_n, sersic_xc, sersic_yc, sersic_ellip, sersic_theta.
    """

    valid = (~mask) & np.isfinite(rms) & (rms != 0)

    chi2 = np.sum(((image[valid] - model[valid]) / rms[valid])**2)

    dof = np.sum(valid) - n_params

    return chi2 / dof

In [61]:
def calculate_RFF(sci_im, sersic_model, rms, mask, flux_auto, flux_radius, x0, y0):
    """
    Formula for RFF from EPOCHS XI eq 4.

    Parameters
    ----------
    sci_im : array
        science image
    sersic_model : array
        Sersic model data
    mask : array
        mask map
    rms : array
        map of background noise
    flux_auto : float
        Flux of galaxy measured through SExtractor
    flux_radius : float
        Half-light radius measurement measured with SExtractor
    x0 : float
        x pixel position of galaxy centre as fit by Pysersic
    y0 : float
        y pixel position of galaxy centre as fit by Pysersic
    """

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    # Uses x0 and y0 from model

    rff_region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated
    # print(np.where(rff_region==True))
    # Remove contaminating sources
    good_pixels = rff_region & (~mask) & np.isfinite(rms) & (rms != 0)
    
    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.nansum(good_pixels)

    background_values = sci_im[~mask.astype(bool)]
    depth_1sig = 1.4826 * np.nanmedian(np.abs(background_values - np.nanmedian(background_values)))

    residuals = sci_im[good_pixels] - sersic_model[good_pixels]

    rff = ( np.nansum(np.abs(residuals)) - 0.8 * depth_1sig * N_pixels)  / flux_auto
    # Must restrict the residual map to the region where RFF is defined (twice flux_radius)

    return rff

In [62]:
def examining_output(galaxy_id, filt, image, rms, mask, psf, source_morphs, centre_id, flux_auto, flux_radius, output_dir, row):

    os.makedirs(output_dir, exist_ok=True)
    
    # morph = source_morphs[centre_id]
    morph = next(m for m in source_morphs if m.label == centre_id)

    ny, nx = image.shape
    y, x = np.mgrid[:ny, :nx]

    model = morph._sersic_model(x,y)
    
    """
    print('BASIC MEASUREMENTS (NON-PARAMETRIC)')
    print('xc_centroid =', morph.xc_centroid)
    print('yc_centroid =', morph.yc_centroid)
    print('ellipticity_centroid =', morph.ellipticity_centroid)
    print('elongation_centroid =', morph.elongation_centroid)
    print('orientation_centroid =', morph.orientation_centroid)
    print('xc_asymmetry =', morph.xc_asymmetry)
    print('yc_asymmetry =', morph.yc_asymmetry)
    print('ellipticity_asymmetry =', morph.ellipticity_asymmetry)
    print('elongation_asymmetry =', morph.elongation_asymmetry)
    print('orientation_asymmetry =', morph.orientation_asymmetry)
    print('rpetro_circ =', morph.rpetro_circ)
    print('rpetro_ellip =', morph.rpetro_ellip)
    print('rhalf_circ =', morph.rhalf_circ)
    print('rhalf_ellip =', morph.rhalf_ellip)
    print('r20 =', morph.r20)
    print('r80 =', morph.r80)
    print('Gini =', morph.gini)
    print('M20 =', morph.m20)
    print('F(G, M20) =', morph.gini_m20_bulge)
    print('S(G, M20) =', morph.gini_m20_merger)
    print('sn_per_pixel =', morph.sn_per_pixel)
    print('C =', morph.concentration)
    print('A =', morph.asymmetry)
    print('S =', morph.smoothness)
    print()
    print('SERSIC MODEL')
    print('sersic_amplitude =', morph.sersic_amplitude)
    print('sersic_rhalf =', morph.sersic_rhalf)
    print('sersic_n =', morph.sersic_n)
    print('sersic_xc =', morph.sersic_xc)
    print('sersic_yc =', morph.sersic_yc)
    print('sersic_ellip =', morph.sersic_ellip)
    print('sersic_theta =', morph.sersic_theta)
    print('sersic_chi2_dof =', morph.sersic_chi2_dof)
    print()
    print('OTHER')
    print('sky_mean =', morph.sky_mean)
    print('sky_median =', morph.sky_median)
    print('sky_sigma =', morph.sky_sigma)
    print('flag =', morph.flag)
    print('flag_sersic =', morph.flag_sersic)
    """
    
    rff = calculate_RFF(image, model, rms, mask, flux_auto, flux_radius, morph.xc_centroid, morph.yc_centroid)

    # red_chi = reduced_chi_squared(image, model, rms, mask, n_params=7)

    red_chi = morph._sersic_chi2 / morph._sersic_num_dof # 7 dof
    
    values = {
        "Gini": morph.gini,
        "M20": morph.m20,
        "F(G,M20)": morph.gini_m20_bulge,
        "S(G,M20)": morph.gini_m20_merger,
        "C": morph.concentration,
        "A": morph.asymmetry,
        "S": morph.smoothness,
        "sersic_n": morph.sersic_n,
        "rpetro_circ": morph.rpetro_circ,
        "rpetro_ellip": morph.rpetro_ellip,
        "rhalf_circ": morph.rhalf_circ,
        "rhalf_ellip": morph.rhalf_ellip,
        "flag": morph.flag,
        "flag_sersic": morph.flag_sersic,
        "RFF": rff,
        "red_chi": red_chi,
    }

    for key, value in values.items():
        row[f"{filt}_{key}"] = value

    fig = make_figure(morph)
    fig_path = os.path.join(output_dir, f"{galaxy_id}_{filt}_statmorph.png")
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)

    return row

In [63]:
def run_everything():

    rows = []

    for i in range(len(GALAXY_ID)):

        row = {
            "SURVEY_ID": GALAXY_ID[i],
            "REDSHIFT": TABLE["REDSHIFT"][i],
        }

        for filt in FILTERS:

            try:

                science_fits_path = os.path.join(CUTOUTS_DIR, f"{GALAXY_ID[i]}/{filt}.fits" )    
                
                image, mask, rms, segmap = load_science_data(science_fits_path)

                ny, nx = segmap.shape
                cy, cx = ny // 2, nx // 2
                centre_id = segmap[cy, cx]

                # Path to the PSF FITS file, psf will be cropped to match/fit the science image.
                psf_fits_path = os.path.join(PSF_DIR, f"{SURVEY[i]}/{filt}_psf_norm.fits")
                psf = load_and_crop_psf(psf_fits_path, image.shape, psf_ext=0)
                # psf = read_fits_path(psf_fits_path, ext=0)

                # Output directory for plots/results
                output_dir = os.path.join(OUTPUT_DIR, f"{GALAXY_ID[i]}")

                flux_auto = f"FLUX_AUTO_{filt}"
                flux_radius = f"FLUX_RADIUS_{filt}"

                source_morphs = statmorph_model(image, segmap, psf, rms, mask) # I dont use my own mask, i let statmorph make one
                
                row = examining_output(GALAXY_ID[i], filt, image, rms, mask, psf, source_morphs, 
                                       centre_id, TABLE[flux_auto][i], TABLE[flux_radius][i],
                                       output_dir, row)
                
            except Exception as e:
                print(e)
                continue

        rows.append(row)
    
    statmorph_table = Table(rows=rows)
    table_path = os.path.join(OUTPUT_DIR, "Statmorph_table.fits")
    statmorph_table.write(table_path, format="fits", overwrite=True)

In [64]:
if __name__ == "__main__":
    run_everything()

Time: 4.38442 s.


Time: 5.23405 s.


Time: 7.01996 s.
Time: 3.16293 s.


Time: 10.705 s.
Time: 4.37328 s.


Time: 3.71107 s.


...

Time: 3.81704 s.


Time: 4.50484 s.


Time: 2.51865 s.


Time: 4.94938 s.


Time: 4.6073 s.
